<a href="https://www.kaggle.com/code/avikdas567/march-machine-learning-mania-2026?scriptVersionId=300156158" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/march-machine-learning-mania-2026"

# Build team-season ratings

def build_team_ratings(prefix):
    df = pd.read_csv(f"{DATA_PATH}/{prefix}RegularSeasonCompactResults.csv")

    win = df[["Season","DayNum","WTeamID","WScore","LScore","LTeamID"]].copy()
    win.columns = ["Season","DayNum","TeamID","ScoreFor","ScoreAgainst","OppID"]
    win["Win"] = 1

    lose = df[["Season","DayNum","LTeamID","LScore","WScore","WTeamID"]].copy()
    lose.columns = ["Season","DayNum","TeamID","ScoreFor","ScoreAgainst","OppID"]
    lose["Win"] = 0

    games = pd.concat([win, lose], ignore_index=True)
    games["Margin"] = games["ScoreFor"] - games["ScoreAgainst"]

    base = games.groupby(["Season","TeamID"]).agg(
        games=("Win","count"),
        win_rate=("Win","mean"),
        avg_margin=("Margin","mean")
    ).reset_index()

    opp_wr = base[["Season","TeamID","win_rate"]].rename(
        columns={"TeamID":"OppID","win_rate":"opp_win_rate"}
    )
    games = games.merge(opp_wr, on=["Season","OppID"], how="left")
    sos = games.groupby(["Season","TeamID"])["opp_win_rate"].mean().reset_index()
    base = base.merge(sos, on=["Season","TeamID"], how="left")
    base["opp_win_rate"] = base["opp_win_rate"].fillna(0.5)

    recent = games[games["DayNum"] >= games["DayNum"].max() - 30]
    recent_form = recent.groupby(["Season","TeamID"])["Win"].mean().reset_index()
    recent_form.columns = ["Season","TeamID","recent_win_rate"]
    base = base.merge(recent_form, on=["Season","TeamID"], how="left")
    base["recent_win_rate"] = base["recent_win_rate"].fillna(base["win_rate"])

    return base

# Training data

def build_training_data(prefix):
    results = pd.read_csv(f"{DATA_PATH}/{prefix}RegularSeasonCompactResults.csv")
    ratings = build_team_ratings(prefix)

    df = results.merge(
        ratings, left_on=["Season","WTeamID"], right_on=["Season","TeamID"]
    ).merge(
        ratings, left_on=["Season","LTeamID"], right_on=["Season","TeamID"],
        suffixes=("_W","_L")
    )

    X = pd.DataFrame({
        "win_rate_diff": df["win_rate_W"] - df["win_rate_L"],
        "margin_diff": df["avg_margin_W"] - df["avg_margin_L"],
        "sos_diff": df["opp_win_rate_W"] - df["opp_win_rate_L"],
        "recent_diff": df["recent_win_rate_W"] - df["recent_win_rate_L"],
    })

    y = np.ones(len(df))
    seasons = df["Season"]

    X = pd.concat([X, -X], ignore_index=True)
    y = np.concatenate([y, np.zeros(len(df))])
    seasons = pd.concat([seasons, seasons], ignore_index=True)

    return X, y, seasons, ratings

# Load MEN + WOMEN

X_m, y_m, s_m, ratings_m = build_training_data("M")
X_w, y_w, s_w, ratings_w = build_training_data("W")

X = pd.concat([X_m, X_w], ignore_index=True)
y = np.concatenate([y_m, y_w])
seasons = pd.concat([s_m, s_w], ignore_index=True)

mask = seasons.between(2022, 2025)
X_train = X[mask]
y_train = y[mask]

# Train model

model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(C=0.3, max_iter=2000))
])
model.fit(X_train, y_train)

# Submission

sub = pd.read_csv(f"{DATA_PATH}/SampleSubmissionStage1.csv")
ids = sub["ID"].str.split("_", expand=True)
sub["Season"] = ids[0].astype(int)
sub["Team1"] = ids[1].astype(int)
sub["Team2"] = ids[2].astype(int)

def predict(sub_df, ratings):
    df = sub_df.merge(
        ratings, left_on=["Season","Team1"], right_on=["Season","TeamID"], how="left"
    ).merge(
        ratings, left_on=["Season","Team2"], right_on=["Season","TeamID"],
        suffixes=("_1","_2"), how="left"
    )

    for c in ["win_rate","avg_margin","opp_win_rate","recent_win_rate"]:
        df[f"{c}_1"] = df[f"{c}_1"].fillna(0.5)
        df[f"{c}_2"] = df[f"{c}_2"].fillna(0.5)

    Xp = pd.DataFrame({
        "win_rate_diff": df["win_rate_1"] - df["win_rate_2"],
        "margin_diff": df["avg_margin_1"] - df["avg_margin_2"],
        "sos_diff": df["opp_win_rate_1"] - df["opp_win_rate_2"],
        "recent_diff": df["recent_win_rate_1"] - df["recent_win_rate_2"],
    })

    p = model.predict_proba(Xp)[:,1]

    p = 0.90 * p + 0.10 * 0.5
    return p

men_mask = sub["Team1"] < 2000
women_mask = sub["Team1"] >= 3000

sub.loc[men_mask, "Pred"] = predict(sub[men_mask], ratings_m)
sub.loc[women_mask, "Pred"] = predict(sub[women_mask], ratings_w)

sub["Pred"] = sub["Pred"].clip(0.05, 0.95)

submission = sub[["ID","Pred"]]
submission.to_csv("submission.csv", index=False)

print("submission.csv created")
submission.head()

submission.csv created


,ID,Pred
0,2022_1101_1102,0.762033
1,2022_1101_1103,0.436558
2,2022_1101_1104,0.144483
3,2022_1101_1105,0.905628
4,2022_1101_1106,0.909719
